# Logistic regression results

This notebook reads the completed experiment in `reports/logistic_regression`; running its cells does not train models.

A/B/C/D compare original or enhanced features with observed or augmented training. Error bars show the sample SD of ten outer-fold means. The team can combine these predictions with the other classifiers for the final comparison.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks': PROJECT = PROJECT.parent
REPORT = PROJECT / 'reports/logistic_regression'
READY = (REPORT / 'completion.json').exists()
if READY:
    completion = json.loads((REPORT / 'completion.json').read_text())
    display(completion)
else:
    print('Complete results are not available yet; do not interpret partial runs as final results.')

## Observed-data primary results

Macro-F1 balances the two class F1 scores. Sensitivity measures AFIB detection; specificity measures correctly excluded non-AFIB records. AFIB absence does not establish health. Training class weights mean raw predicted probabilities need not be calibrated to population prevalence.

In [ ]:
if READY:
    summary = pd.read_csv(REPORT / 'summary.csv')
    primary = summary[(summary.threshold_policy == 'fixed_0.50') & summary.variant.isin(['A','B','C','D'])]
    observed = primary[(primary.scenario == 'observed') & primary.metric.isin(['macro_f1','sensitivity','specificity','auroc','average_precision','brier'])]
    display(observed[['variant','metric','mean','sd','n_folds','n_defined']])
    display(pd.read_csv(REPORT / 'majority_baseline.csv'))

## Matched stress conditions

Compare absolute scores and changes from observed input within the same outer folds. A smaller drop is useful only when the remaining performance is also adequate.

In [ ]:
if READY:
    from IPython.display import Image
    for name in ['macro_f1','sensitivity','brier']:
        display(Image(filename=str(REPORT / f'{name}.png')))
    display(pd.read_csv(REPORT / 'paired_method_differences.csv').query("metric == 'macro_f1'"))

## Hyperparameter selection and convergence

C is selected only on inner observed macro-F1. One endpoint-directed refinement is allowed; outcomes are never changed to manufacture a middle winner. Check whether the middle was most often selected and discuss remaining endpoints.

In [ ]:
if READY:
    selection = pd.read_csv(REPORT / 'selected_hyperparameters.csv')
    display(selection.groupby(['variant','C']).size().rename('outer_folds').reset_index())
    display(selection.groupby('variant').selected_middle.agg(['sum','count']))
    print('All fits converged:', completion['all_fits_converged'])

## Controls and separate filling comparison

Availability-only controls remove constructed values but retain their missingness indicators. Duplicate controls preserve original total weights. Ridge recovery scores only artificially hidden, originally observed eligible fields; natural gaps have no known truth. Filling error reductions alone do not establish better classification.

In [ ]:
if READY:
    display(pd.read_csv(REPORT / 'recovery_summary.csv'))
    display(pd.read_csv(REPORT / 'paired_method_differences.csv').query("comparison == 'ridge minus A' and metric in ['macro_f1','sensitivity','specificity','brier']"))
    print('Detailed subgroup, patient-weighted, reliability and confusion-count tables are in', REPORT)